# Project: Conway's Game of Life

Simulate a cellular automaton, first in pure Python, then with numpy, test it, and animate it with matplotlib.

The full explanations are in [documentation_tests_game_of_life.md](documentation_tests_game_of_life.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## The rules

The universe is a grid of cells, **alive** (1) or **dead** (0). Each cell has
**8 neighbours** (diagonals included). At each generation, all the cells are
updated at the same time:

* a dead cell with **exactly 3** living neighbours becomes alive (birth);
* a living cell with **2 or 3** living neighbours stays alive (survival);
* all the other cells die, or stay dead.

Cells outside the grid are considered dead.

## Step 1: counting the neighbours

The grid is a list of lists: `grid[i][j]` is the cell at row `i`, column `j`.
Try to write `count_neighbours` yourself from its docstring before running
the cell!

In [ ]:
%%writefile life.py
"""Conway's Game of Life.

Two implementations of the same rules: a pure Python one, working on lists
of lists, and a numpy one, working on 2D arrays. Cells outside the grid are
considered dead.
"""

import numpy as np


def count_neighbours(grid, i, j):
    """Count the living neighbours of a cell.

    Parameters
    ----------
    grid : list of lists of int
        The grid, with 1 for a living cell and 0 for a dead cell.
    i, j : int
        Row and column of the cell.

    Returns
    -------
    int
        Number of living cells among the 8 neighbours (0 to 8).

    Examples
    --------
    >>> count_neighbours([[1, 1, 0], [0, 1, 0], [0, 0, 0]], 1, 1)
    2
    """
    n_rows, n_cols = len(grid), len(grid[0])
    count = 0
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue  # the cell itself is not a neighbour
            ni, nj = i + di, j + dj
            if 0 <= ni < n_rows and 0 <= nj < n_cols:
                count += grid[ni][nj]
    return count

First tests: on a full grid, we know how many neighbours each cell has.

In [ ]:
%%writefile test_life.py
"""Tests for life.py, to be run with pytest."""

from life import count_neighbours


def test_count_neighbours_full_grid():
    grid = [[1, 1, 1],
            [1, 1, 1],
            [1, 1, 1]]
    assert count_neighbours(grid, 1, 1) == 8   # center
    assert count_neighbours(grid, 0, 0) == 3   # corner
    assert count_neighbours(grid, 0, 1) == 5   # edge


def test_count_neighbours_empty_grid():
    grid = [[0, 0, 0],
            [0, 0, 0],
            [0, 0, 0]]
    assert count_neighbours(grid, 1, 1) == 0

In [ ]:
import sys
!{sys.executable} -m pytest -q

## Step 2: computing the next generation

`step` builds a **new** grid: all the cells are updated at the same time, so
we must not modify the grid while reading it. The cell below rewrites
`life.py` with the new function added.

In [ ]:
%%writefile life.py
"""Conway's Game of Life.

Two implementations of the same rules: a pure Python one, working on lists
of lists, and a numpy one, working on 2D arrays. Cells outside the grid are
considered dead.
"""

import numpy as np


def count_neighbours(grid, i, j):
    """Count the living neighbours of a cell.

    Parameters
    ----------
    grid : list of lists of int
        The grid, with 1 for a living cell and 0 for a dead cell.
    i, j : int
        Row and column of the cell.

    Returns
    -------
    int
        Number of living cells among the 8 neighbours (0 to 8).

    Examples
    --------
    >>> count_neighbours([[1, 1, 0], [0, 1, 0], [0, 0, 0]], 1, 1)
    2
    """
    n_rows, n_cols = len(grid), len(grid[0])
    count = 0
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue  # the cell itself is not a neighbour
            ni, nj = i + di, j + dj
            if 0 <= ni < n_rows and 0 <= nj < n_cols:
                count += grid[ni][nj]
    return count


def step(grid):
    """Compute the next generation (pure Python version).

    Parameters
    ----------
    grid : list of lists of int
        The current generation.

    Returns
    -------
    list of lists of int
        The next generation. The input grid is not modified.
    """
    new_grid = []
    for i in range(len(grid)):
        new_row = []
        for j in range(len(grid[0])):
            n = count_neighbours(grid, i, j)
            if grid[i][j] == 1 and n in (2, 3):
                new_row.append(1)   # survival
            elif grid[i][j] == 0 and n == 3:
                new_row.append(1)   # birth
            else:
                new_row.append(0)   # death or stays dead
        new_grid.append(new_row)
    return new_grid

The block must not change, the blinker must oscillate, and the input must not be modified:

In [ ]:
%%writefile test_life.py
"""Tests for life.py, to be run with pytest."""

from life import count_neighbours, step


def test_count_neighbours_full_grid():
    grid = [[1, 1, 1],
            [1, 1, 1],
            [1, 1, 1]]
    assert count_neighbours(grid, 1, 1) == 8   # center
    assert count_neighbours(grid, 0, 0) == 3   # corner
    assert count_neighbours(grid, 0, 1) == 5   # edge


def test_count_neighbours_empty_grid():
    grid = [[0, 0, 0],
            [0, 0, 0],
            [0, 0, 0]]
    assert count_neighbours(grid, 1, 1) == 0


def test_block_is_stable():
    block = [[0, 0, 0, 0],
             [0, 1, 1, 0],
             [0, 1, 1, 0],
             [0, 0, 0, 0]]
    assert step(block) == block


def test_blinker_oscillates():
    vertical = [[0, 0, 0, 0, 0],
                [0, 0, 1, 0, 0],
                [0, 0, 1, 0, 0],
                [0, 0, 1, 0, 0],
                [0, 0, 0, 0, 0]]
    horizontal = [[0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0],
                  [0, 1, 1, 1, 0],
                  [0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0]]
    assert step(vertical) == horizontal
    assert step(horizontal) == vertical


def test_step_does_not_modify_input():
    grid = [[0, 1, 0],
            [0, 1, 0],
            [0, 1, 0]]
    copy = [row[:] for row in grid]
    step(grid)
    assert grid == copy

In [ ]:
import sys
!{sys.executable} -m pytest -q

**See a test fail**: in the `life.py` cell above, replace `n in (2, 3)`
by `n == 2`, run it again, then run the tests. Read the output, then fix the
bug.

Note: after modifying `life.py`, the tests (run by pytest in a separate
process) see the new version immediately, but this notebook keeps the old one
in memory. The next cell forces Python to reload the module.

In [ ]:
import importlib
import life
importlib.reload(life)

## Step 3: watching it live (in text mode)

In [ ]:
from life import step


def show(grid):
    """Print the grid, with # for living cells and . for dead cells."""
    for row in grid:
        print("".join("#" if cell else "." for cell in row))
    print()


glider = [[0, 1, 0, 0, 0, 0],
          [0, 0, 1, 0, 0, 0],
          [1, 1, 1, 0, 0, 0],
          [0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0]]

grid = glider
for generation in range(5):
    show(grid)
    grid = step(grid)

After 4 generations, the glider has its initial shape, shifted by one cell down and to the right.

## Step 4: a numpy version

Add a border of dead cells (`np.pad`), take 8 slices shifted by one cell in
each direction, and add them: this gives the number of living neighbours of
**all** the cells at once.

In [ ]:
%%writefile life.py
"""Conway's Game of Life.

Two implementations of the same rules: a pure Python one, working on lists
of lists, and a numpy one, working on 2D arrays. Cells outside the grid are
considered dead.
"""

import numpy as np


def count_neighbours(grid, i, j):
    """Count the living neighbours of a cell.

    Parameters
    ----------
    grid : list of lists of int
        The grid, with 1 for a living cell and 0 for a dead cell.
    i, j : int
        Row and column of the cell.

    Returns
    -------
    int
        Number of living cells among the 8 neighbours (0 to 8).

    Examples
    --------
    >>> count_neighbours([[1, 1, 0], [0, 1, 0], [0, 0, 0]], 1, 1)
    2
    """
    n_rows, n_cols = len(grid), len(grid[0])
    count = 0
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue  # the cell itself is not a neighbour
            ni, nj = i + di, j + dj
            if 0 <= ni < n_rows and 0 <= nj < n_cols:
                count += grid[ni][nj]
    return count


def step(grid):
    """Compute the next generation (pure Python version).

    Parameters
    ----------
    grid : list of lists of int
        The current generation.

    Returns
    -------
    list of lists of int
        The next generation. The input grid is not modified.
    """
    new_grid = []
    for i in range(len(grid)):
        new_row = []
        for j in range(len(grid[0])):
            n = count_neighbours(grid, i, j)
            if grid[i][j] == 1 and n in (2, 3):
                new_row.append(1)   # survival
            elif grid[i][j] == 0 and n == 3:
                new_row.append(1)   # birth
            else:
                new_row.append(0)   # death or stays dead
        new_grid.append(new_row)
    return new_grid


def step_numpy(Z):
    """Compute the next generation (numpy version).

    Parameters
    ----------
    Z : numpy.ndarray
        2D array of 0 and 1, the current generation.

    Returns
    -------
    numpy.ndarray
        The next generation, with the same shape as Z.
    """
    P = np.pad(Z, 1)  # add a border of dead cells
    N = (P[:-2, :-2] + P[:-2, 1:-1] + P[:-2, 2:] +
         P[1:-1, :-2]               + P[1:-1, 2:] +
         P[2:, :-2]  + P[2:, 1:-1]  + P[2:, 2:])
    birth = (Z == 0) & (N == 3)
    survival = (Z == 1) & ((N == 2) | (N == 3))
    return (birth | survival).astype(int)

The numpy version must give exactly the same result as the Python version, on many random grids:

In [ ]:
%%writefile test_life.py
"""Tests for life.py, to be run with pytest."""

import numpy as np

from life import count_neighbours, step, step_numpy


def test_count_neighbours_full_grid():
    grid = [[1, 1, 1],
            [1, 1, 1],
            [1, 1, 1]]
    assert count_neighbours(grid, 1, 1) == 8   # center
    assert count_neighbours(grid, 0, 0) == 3   # corner
    assert count_neighbours(grid, 0, 1) == 5   # edge


def test_count_neighbours_empty_grid():
    grid = [[0, 0, 0],
            [0, 0, 0],
            [0, 0, 0]]
    assert count_neighbours(grid, 1, 1) == 0


def test_block_is_stable():
    block = [[0, 0, 0, 0],
             [0, 1, 1, 0],
             [0, 1, 1, 0],
             [0, 0, 0, 0]]
    assert step(block) == block


def test_blinker_oscillates():
    vertical = [[0, 0, 0, 0, 0],
                [0, 0, 1, 0, 0],
                [0, 0, 1, 0, 0],
                [0, 0, 1, 0, 0],
                [0, 0, 0, 0, 0]]
    horizontal = [[0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0],
                  [0, 1, 1, 1, 0],
                  [0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0]]
    assert step(vertical) == horizontal
    assert step(horizontal) == vertical


def test_step_does_not_modify_input():
    grid = [[0, 1, 0],
            [0, 1, 0],
            [0, 1, 0]]
    copy = [row[:] for row in grid]
    step(grid)
    assert grid == copy


def test_numpy_matches_python():
    rng = np.random.default_rng(0)
    for _ in range(10):
        Z = rng.integers(0, 2, size=(20, 30))
        expected = step(Z.tolist())
        assert step_numpy(Z).tolist() == expected

In [ ]:
import sys
!{sys.executable} -m pytest -q --doctest-modules life.py test_life.py

Was it worth it? Compare the speed of the two versions on a 100×100 grid:

In [ ]:
import numpy as np
import importlib
import life
importlib.reload(life)

Z = np.random.default_rng(0).integers(0, 2, size=(100, 100))
L = Z.tolist()
%timeit life.step(L)
%timeit life.step_numpy(Z)

## Step 5: animating with matplotlib

In a notebook, an animation is displayed with `to_jshtml()`, which builds a
small player with play/pause buttons. (In the lesson, the same animation is
run from the shell by the script `run_life.py`.)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

from life import step_numpy

rng = np.random.default_rng(1)
Z = (rng.random((60, 60)) < 0.2).astype(int)

fig, ax = plt.subplots(figsize=(5, 5))
image = ax.imshow(Z, cmap="gray_r", interpolation="nearest")
ax.set_xticks([])
ax.set_yticks([])


def update(frame):
    global Z
    Z = step_numpy(Z)
    image.set_data(Z)
    ax.set_title(f"Generation {frame + 1} - {Z.sum()} living cells")
    return [image]


animation = FuncAnimation(fig, update, frames=100, interval=80)
plt.close(fig)          # avoid displaying an extra still image
HTML(animation.to_jshtml())

In a notebook, `update` modifies the variable `Z` of the notebook itself,
hence `global Z` (in the script, `Z` belongs to `main`, hence `nonlocal Z`).

The script version, with command-line options, can be written and run from
here too:

In [ ]:
%%writefile run_life.py
"""Run and display Conway's Game of Life."""

import argparse

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

from life import step_numpy


def main():
    parser = argparse.ArgumentParser(description="Conway's Game of Life.")
    parser.add_argument("--size", type=int, default=100, help="grid size")
    parser.add_argument("--density", type=float, default=0.2,
                        help="initial proportion of living cells")
    parser.add_argument("--seed", type=int, default=None, help="random seed")
    parser.add_argument("--save", default=None,
                        help="save the animation to this file (e.g. life.gif)")
    args = parser.parse_args()

    rng = np.random.default_rng(args.seed)
    Z = (rng.random((args.size, args.size)) < args.density).astype(int)

    fig, ax = plt.subplots(figsize=(6, 6))
    image = ax.imshow(Z, cmap="gray_r", interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])

    def update(frame):
        nonlocal Z
        Z = step_numpy(Z)
        image.set_data(Z)
        ax.set_title(f"Generation {frame + 1} - {Z.sum()} living cells")
        return [image]

    animation = FuncAnimation(fig, update, frames=200, interval=50)
    if args.save:
        animation.save(args.save, fps=20)
        print("Saved", args.save)
    else:
        plt.show()


if __name__ == "__main__":
    main()

In [ ]:
%run run_life.py --size 50 --seed 1 --save life.gif

Open `life.gif` from the file browser on the left (double-click).

## Going further

**1.** Add a test checking that after 4 generations, the glider has moved by one cell down and one cell to the right.

<details><summary>Answer</summary>

```python
def test_glider_moves():
    glider = [[0, 1, 0, 0, 0, 0],
              [0, 0, 1, 0, 0, 0],
              [1, 1, 1, 0, 0, 0],
              [0, 0, 0, 0, 0, 0],
              [0, 0, 0, 0, 0, 0],
              [0, 0, 0, 0, 0, 0]]
    grid = glider
    for _ in range(4):
        grid = step(grid)
    expected = [[0] * 6] + [[0] + row[:-1] for row in glider[:-1]]
    assert grid == expected
```

</details>

In [ ]:
# Your code here

**2.** Plot the number of living cells as a function of the generation, for a random 100×100 grid over 300 generations.

<details><summary>Answer</summary>

```python
rng = np.random.default_rng(1)
Z = (rng.random((100, 100)) < 0.2).astype(int)
population = []
for generation in range(300):
    population.append(Z.sum())
    Z = step_numpy(Z)
plt.plot(population)
plt.xlabel("Generation")
plt.ylabel("Living cells")
```

</details>

In [ ]:
# Your code here

**3.** Write a version `step_torus` where the grid wraps around (a cell on the right edge has neighbours on the left edge). Hint: `np.roll`.

<details><summary>Answer</summary>

```python
def step_torus(Z):
    N = sum(np.roll(np.roll(Z, di, axis=0), dj, axis=1)
            for di in (-1, 0, 1) for dj in (-1, 0, 1)
            if (di, dj) != (0, 0))
    birth = (Z == 0) & (N == 3)
    survival = (Z == 1) & ((N == 2) | (N == 3))
    return (birth | survival).astype(int)
```

On a 6×6 torus, a glider gets back to its initial position after 24 generations.

</details>

In [ ]:
# Your code here

**4.** Generalize `step_numpy` into `step_rule(Z, birth=(3,), survival=(2, 3))` and try *HighLife* (`birth=(3, 6)`).

<details><summary>Answer</summary>

```python
def step_rule(Z, birth=(3,), survival=(2, 3)):
    P = np.pad(Z, 1)
    N = (P[:-2, :-2] + P[:-2, 1:-1] + P[:-2, 2:] +
         P[1:-1, :-2]               + P[1:-1, 2:] +
         P[2:, :-2]  + P[2:, 1:-1]  + P[2:, 2:])
    born = (Z == 0) & np.isin(N, birth)
    survive = (Z == 1) & np.isin(N, survival)
    return (born | survive).astype(int)
```

Check it with a test: with the default rule, it must give the same result as `step_numpy`.

</details>

In [ ]:
# Your code here